In [1]:
# ============================================
# K-Means Clustering with NLP
# Dataset: latex-typesetting-with-excel.csv
# ============================================

# Install packages if needed
# !pip install pandas numpy matplotlib seaborn scikit-learn nltk wordcloud

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import nltk
from nltk.corpus import stopwords
import re

# Download stopwords (first run only)
nltk.download('stopwords')

# -----------------------------
# Load Dataset
# -----------------------------
df = pd.read_csv("latex-typesetting-with-excel.csv")

print("Dataset Shape:", df.shape)
print(df.head())

# -----------------------------
# Dataset Information
# -----------------------------
print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

# -----------------------------
# Detect Text Columns
# -----------------------------
text_columns = df.select_dtypes(include=['object']).columns.tolist()

print("\nDetected Text Columns:")
print(text_columns)

if len(text_columns) == 0:
    raise ValueError("No text columns found for NLP analysis.")

# Merge text columns
df["combined_text"] = (
    df[text_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

# -----------------------------
# Text Cleaning Function
# -----------------------------
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df["clean_text"] = df["combined_text"].apply(clean_text)

print(df[["clean_text"]].head())

# -----------------------------
# TF-IDF Feature Extraction
# -----------------------------
vectorizer = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1,2)
)

X = vectorizer.fit_transform(df["clean_text"])

print("TF-IDF Matrix Shape:", X.shape)

# -----------------------------
# Elbow Method
# -----------------------------
inertia = []

cluster_range = range(2,11)

for k in cluster_range:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    model.fit(X)
    inertia.append(model.inertia_)

plt.figure(figsize=(8,5))
plt.plot(cluster_range, inertia, marker='o')
plt.title("Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.grid(True)
plt.show()

# -----------------------------
# Train Final Model
# -----------------------------
k = 5

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X)

df["Cluster"] = clusters

print(df[["Cluster"]].head())

# -----------------------------
# Silhouette Score
# -----------------------------
score = silhouette_score(X, clusters)

print("\nSilhouette Score:", round(score,3))

# -----------------------------
# Top Keywords Per Cluster
# -----------------------------
terms = vectorizer.get_feature_names_out()

print("\nTop Keywords Per Cluster\n")

order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

for i in range(k):
    keywords = [terms[ind] for ind in order_centroids[i, :10]]
    print(f"Cluster {i}")
    print(", ".join(keywords))
    print()

# -----------------------------
# PCA Visualization
# -----------------------------
pca = PCA(n_components=2)

reduced = pca.fit_transform(X.toarray())

plt.figure(figsize=(9,6))

plt.scatter(
    reduced[:,0],
    reduced[:,1],
    c=df["Cluster"],
    cmap="tab10",
    s=50
)

plt.title("K-Means Clusters (PCA Projection)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.colorbar(label="Cluster")
plt.show()

# -----------------------------
# Cluster Sizes
# -----------------------------
print("\nCluster Counts")

print(df["Cluster"].value_counts().sort_index())

# -----------------------------
# Save Results
# -----------------------------
output_file = "latex-typesetting-with-excel_clustered.csv"

df.to_csv(output_file, index=False)

print(f"\nClustered dataset saved as: {output_file}")

# -----------------------------
# Display Sample Rows
# -----------------------------
for i in sorted(df["Cluster"].unique()):
    print(f"\nCluster {i} Sample")
    print(df[df["Cluster"] == i].head())

Dataset Shape: (4, 4)
   X1  Y1  X2   Y2
0   1   1   1  0.5
1   2   4   2  2.0
2   3   9   3  4.5
3   4  16   8  8.0

Column Names:
['X1', 'Y1', 'X2', 'Y2']

Data Types:
X1      int64
Y1      int64
X2      int64
Y2    float64
dtype: object

Detected Text Columns:
[]


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


ValueError: No text columns found for NLP analysis.